# Gavia, ecoSUB, and Sonardyne/Ranger2 position comparison pipeline

This notebook ingests:

- **Gavia** zipped XML log files
- **ecoSUB** CSV files
- **Sonardyne Ranger2** CSV exports

It normalises all position data onto a shared schema, aligns vehicle positions to tracked positions by timestamp, calculates separation/error metrics, and creates plots for QC and comparison.

## Expected output

The core comparison table produced by this notebook is:

| field | description |
|---|---|
| `source_vehicle` | e.g. `gavia` or `ecosub` |
| `time_utc` | vehicle timestamp |
| `track_time_utc` | nearest Ranger2/Sonardyne timestamp |
| `dt_seconds` | time offset between vehicle and tracking fix |
| `lat`, `lon` | vehicle position |
| `track_lat`, `track_lon` | tracked position |
| `east_error_m`, `north_error_m` | local ENU difference |
| `horizontal_error_m` | horizontal separation |
| `depth_error_m` | optional, if both depth columns exist |

## Folder layout

Create this structure next to the notebook, or change the paths in the config cell:

```text
project/
  data/
    gavia_zip/
      *.zip
    ecosub_csv/
      *.csv
    sonardyne_csv/
      *.csv
  output/
```

In [ ]:
# Optional: uncomment in a fresh environment
# %pip install pandas numpy matplotlib pyproj lxml

from pathlib import Path
import zipfile
import shutil
import re
import math
import warnings
import xml.etree.ElementTree as ET
from datetime import timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from pyproj import CRS, Transformer
    HAS_PYPROJ = True
except Exception:
    HAS_PYPROJ = False
    warnings.warn("pyproj not available. Using approximate local metre conversion instead.")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 1. Configuration

Edit these paths and mappings first. The notebook is deliberately schema-tolerant because export column names often vary between missions and software versions.

The mappings below are searched case-insensitively. For example, if an ecoSUB file uses `latitude_deg` instead of `lat`, add it to `column_candidates["lat"]`.

In [ ]:
PROJECT_DIR = Path.cwd()

CONFIG = {
    "paths": {
        "gavia_zip_dir": PROJECT_DIR / "data" / "gavia_zip",
        "gavia_extract_dir": PROJECT_DIR / "data" / "gavia_xml_extracted",
        "ecosub_csv_dir": PROJECT_DIR / "data" / "ecosub_csv",
        "sonardyne_csv_dir": PROJECT_DIR / "data" / "sonardyne_csv",
        "output_dir": PROJECT_DIR / "output",
    },

    # Maximum allowed time separation when matching vehicle position to tracking position
    "merge_tolerance": "5s",

    # If your timestamps are local rather than UTC, set e.g. "Europe/London".
    # The final output is converted to UTC.
    "assume_timezone": "UTC",

    # Candidates are matched after normalising names: lower-case and remove spaces, underscores, hyphens, brackets.
    "column_candidates": {
        "time": [
            "time", "timestamp", "datetime", "date_time", "utc", "utc_time",
            "gps_time", "fix_time", "posix_time", "unix_time", "epoch"
        ],
        "lat": [
            "lat", "latitude", "latitude_deg", "gps_lat", "gpslatitude", "nav_lat", "position_latitude"
        ],
        "lon": [
            "lon", "long", "longitude", "longitude_deg", "gps_lon", "gpslongitude", "nav_lon", "position_longitude"
        ],
        "depth": [
            "depth", "depth_m", "vehicle_depth", "z", "z_m"
        ],
        "altitude": [
            "altitude", "altitude_m", "height_above_seabed", "hab", "range_altitude"
        ],
        "easting": [
            "easting", "x", "x_m", "utm_easting", "east", "local_east"
        ],
        "northing": [
            "northing", "y", "y_m", "utm_northing", "north", "local_north"
        ],
        "vehicle_id": [
            "vehicle", "vehicle_id", "platform", "name", "beacon", "target", "transponder"
        ],
    },

    # Optional filters. Leave as None to keep all rows.
    "vehicle_name_filters": {
        "gavia": None,     # e.g. "Gavia"
        "ecosub": None,    # e.g. "ecoSUB"
        "sonardyne": None, # e.g. "Gavia|ecoSUB|Beacon"
    },
}

for p in CONFIG["paths"].values():
    p.mkdir(parents=True, exist_ok=True)

CONFIG["paths"]

## 2. Utility functions

These helpers handle common problems in field data:

- inconsistent column names
- mixed timestamp formats
- lat/lon in decimal degrees or degrees/minutes strings
- optional local coordinates
- zipped XML extraction
- nested XML flattening

In [ ]:
def normalise_name(name: str) -> str:
    """Normalise a column/tag name for robust matching."""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def find_first_column(df: pd.DataFrame, candidates) -> str | None:
    """Return the first DataFrame column matching any candidate, ignoring punctuation/case."""
    lookup = {normalise_name(c): c for c in df.columns}
    for candidate in candidates:
        key = normalise_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def coerce_numeric(series: pd.Series) -> pd.Series:
    """Convert numbers stored with units, commas, or whitespace into floats."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = (
        series.astype(str)
        .str.replace(",", ".", regex=False)
        .str.extract(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", expand=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")


def parse_lat_lon_value(value):
    """
    Parse decimal degrees or simple degree-minute strings.

    Handles examples like:
      56.123456
      "56 12.345 N"
      "56°12.345'N"
      "005 23.456 W"
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    s = str(value).strip()
    if not s:
        return np.nan

    try:
        return float(s)
    except ValueError:
        pass

    hemi = None
    hemi_match = re.search(r"([NSEW])", s, flags=re.I)
    if hemi_match:
        hemi = hemi_match.group(1).upper()

    nums = re.findall(r"[-+]?\d*\.?\d+", s)
    if not nums:
        return np.nan

    nums = [float(x) for x in nums]
    if len(nums) == 1:
        deg = nums[0]
    elif len(nums) >= 2:
        deg = abs(nums[0]) + nums[1] / 60.0
        if nums[0] < 0:
            deg *= -1
    else:
        return np.nan

    if hemi in ("S", "W"):
        deg = -abs(deg)
    elif hemi in ("N", "E"):
        deg = abs(deg)

    return deg


def parse_time(series: pd.Series, assume_timezone="UTC") -> pd.Series:
    """
    Parse timestamps robustly and return timezone-aware UTC datetimes.

    Also handles numeric epochs in seconds or milliseconds.
    """
    if pd.api.types.is_numeric_dtype(series):
        raw = pd.to_numeric(series, errors="coerce")
        unit = "ms" if raw.dropna().median() > 1e11 else "s"
        return pd.to_datetime(raw, unit=unit, errors="coerce", utc=True)

    dt = pd.to_datetime(series, errors="coerce", utc=False)

    try:
        if getattr(dt.dt, "tz", None) is None:
            dt = dt.dt.tz_localize(assume_timezone, ambiguous="NaT", nonexistent="shift_forward")
        else:
            dt = dt.dt.tz_convert("UTC")
    except Exception:
        dt = pd.to_datetime(series, errors="coerce", utc=True)

    return dt.dt.tz_convert("UTC")


def apply_vehicle_filter(df: pd.DataFrame, source: str) -> pd.DataFrame:
    pattern = CONFIG["vehicle_name_filters"].get(source)
    if not pattern or "vehicle_id" not in df.columns:
        return df
    return df[df["vehicle_id"].astype(str).str.contains(pattern, case=False, regex=True, na=False)].copy()


def standardise_position_df(
    df: pd.DataFrame,
    source: str,
    file_path=None,
    column_candidates=None,
    assume_timezone="UTC",
) -> pd.DataFrame:
    """Standardise any position-like table onto the common schema."""
    column_candidates = column_candidates or CONFIG["column_candidates"]

    original_columns = list(df.columns)
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    found = {key: find_first_column(df, candidates) for key, candidates in column_candidates.items()}

    out = pd.DataFrame()
    out["source"] = source
    out["source_file"] = str(file_path) if file_path is not None else None

    if found["time"]:
        out["time_utc"] = parse_time(df[found["time"]], assume_timezone=assume_timezone)
    else:
        out["time_utc"] = pd.NaT

    if found["lat"]:
        out["lat"] = df[found["lat"]].map(parse_lat_lon_value)
    else:
        out["lat"] = np.nan

    if found["lon"]:
        out["lon"] = df[found["lon"]].map(parse_lat_lon_value)
    else:
        out["lon"] = np.nan

    for key in ["depth", "altitude", "easting", "northing"]:
        if found.get(key):
            out[key] = coerce_numeric(df[found[key]])
        else:
            out[key] = np.nan

    if found["vehicle_id"]:
        out["vehicle_id"] = df[found["vehicle_id"]].astype(str)
    else:
        out["vehicle_id"] = source

    out = apply_vehicle_filter(out, source)
    out = out.dropna(subset=["time_utc"]).sort_values("time_utc").reset_index(drop=True)

    useful = ["time_utc", "lat", "lon", "depth", "altitude", "easting", "northing"]
    if out[useful].notna().sum().sum() == 0:
        warnings.warn(f"No useful position columns found in {file_path}. Original columns: {original_columns[:30]}")

    return out

## 3. Gavia zipped XML ingestion

This section improves the usual unzip → XML → CSV workflow by:

1. safely extracting ZIP contents without overwriting same-named files,
2. recursively flattening XML tags and attributes,
3. writing one CSV per XML file for traceability,
4. loading those CSVs back through the same standardisation path as the other sources.

Because Gavia XML schemas can vary by firmware/log type, the parser keeps **all** flattened fields. You can inspect the intermediate CSVs if the automatic column mapping needs tuning.

In [ ]:
def safe_extract_zip(zip_path: Path, extract_root: Path) -> list[Path]:
    """Extract a zip into a folder named after the zip file and return extracted XML paths."""
    zip_path = Path(zip_path)
    destination = extract_root / zip_path.stem
    destination.mkdir(parents=True, exist_ok=True)

    extracted = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            member_name = Path(member.filename)
            if member_name.is_absolute() or ".." in member_name.parts:
                warnings.warn(f"Skipping suspicious zip member: {member.filename}")
                continue

            target = destination / member_name
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue

            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)

            if target.suffix.lower() == ".xml":
                extracted.append(target)

    return extracted


def strip_xml_namespace(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def flatten_xml_element(element, prefix="") -> dict:
    """Flatten an XML element's attributes, text, and immediate children recursively."""
    row = {}

    tag = strip_xml_namespace(element.tag)
    key_base = f"{prefix}_{tag}" if prefix else tag

    for k, v in element.attrib.items():
        row[f"{key_base}_{strip_xml_namespace(k)}"] = v

    text = (element.text or "").strip()
    if text:
        row[key_base] = text

    child_counts = {}
    for child in list(element):
        child_tag = strip_xml_namespace(child.tag)
        child_counts[child_tag] = child_counts.get(child_tag, 0) + 1
        suffix = child_tag if child_counts[child_tag] == 1 else f"{child_tag}{child_counts[child_tag]}"
        row.update(flatten_xml_element(child, f"{key_base}_{suffix}"))

    return row


def xml_to_flat_dataframe(xml_path: Path) -> pd.DataFrame:
    """
    Convert XML to a flat dataframe.

    Strategy:
    - parse the XML tree,
    - flatten every leaf-bearing element,
    - keep rows that contain more than one field.
    """
    xml_path = Path(xml_path)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []
    for elem in root.iter():
        flat = flatten_xml_element(elem)
        if len(flat) > 1:
            rows.append(flat)

    if not rows:
        rows = [flatten_xml_element(root)]

    df = pd.DataFrame(rows)
    df["xml_source_file"] = str(xml_path)
    return df


def process_gavia_zips_to_csv(
    zip_dir: Path,
    extract_dir: Path,
    output_csv_dir: Path,
    force: bool = False
) -> list[Path]:
    """Extract all Gavia ZIPs, flatten XML files, and save intermediate CSVs."""
    zip_dir = Path(zip_dir)
    extract_dir = Path(extract_dir)
    output_csv_dir = Path(output_csv_dir)
    output_csv_dir.mkdir(parents=True, exist_ok=True)

    csv_paths = []

    for zip_path in sorted(zip_dir.glob("*.zip")):
        xml_paths = safe_extract_zip(zip_path, extract_dir)

        for xml_path in xml_paths:
            rel = xml_path.relative_to(extract_dir)
            csv_name = "__".join(rel.with_suffix("").parts) + ".csv"
            csv_path = output_csv_dir / csv_name

            if csv_path.exists() and not force:
                csv_paths.append(csv_path)
                continue

            try:
                df = xml_to_flat_dataframe(xml_path)
                df.to_csv(csv_path, index=False)
                csv_paths.append(csv_path)
            except Exception as exc:
                warnings.warn(f"Failed to process {xml_path}: {exc}")

    return csv_paths


GAVIA_FLAT_CSV_DIR = CONFIG["paths"]["output_dir"] / "gavia_flat_csv"

gavia_flat_csvs = process_gavia_zips_to_csv(
    CONFIG["paths"]["gavia_zip_dir"],
    CONFIG["paths"]["gavia_extract_dir"],
    GAVIA_FLAT_CSV_DIR,
    force=False,
)

print(f"Created/found {len(gavia_flat_csvs)} flattened Gavia CSV files.")
gavia_flat_csvs[:5]

## 4. Load all source datasets

For the Gavia intermediate CSVs, the automatic parser may initially produce several candidate rows from each XML file. The QC cells below help identify which flattened fields contain navigation records.

In [ ]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    """Read CSV with fallback encodings and separator sniffing."""
    path = Path(path)
    for encoding in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=encoding)
        except Exception:
            pass
    return pd.read_csv(path)


def load_standardised_csvs(csv_dir: Path, source: str) -> pd.DataFrame:
    csv_dir = Path(csv_dir)
    rows = []

    for path in sorted(csv_dir.glob("*.csv")):
        try:
            raw = read_csv_flexible(path)
            std = standardise_position_df(
                raw,
                source=source,
                file_path=path,
                column_candidates=CONFIG["column_candidates"],
                assume_timezone=CONFIG["assume_timezone"],
            )
            rows.append(std)
        except Exception as exc:
            warnings.warn(f"Could not load {path}: {exc}")

    if rows:
        return pd.concat(rows, ignore_index=True).sort_values("time_utc").reset_index(drop=True)

    return pd.DataFrame(columns=[
        "source", "source_file", "time_utc", "lat", "lon", "depth",
        "altitude", "easting", "northing", "vehicle_id"
    ])


gavia = load_standardised_csvs(GAVIA_FLAT_CSV_DIR, "gavia")
ecosub = load_standardised_csvs(CONFIG["paths"]["ecosub_csv_dir"], "ecosub")
sonardyne = load_standardised_csvs(CONFIG["paths"]["sonardyne_csv_dir"], "sonardyne")

print("Rows loaded:")
print(f"  Gavia:     {len(gavia):,}")
print(f"  ecoSUB:    {len(ecosub):,}")
print(f"  Sonardyne: {len(sonardyne):,}")

display(gavia.head())
display(ecosub.head())
display(sonardyne.head())

## 5. QC: inspect missingness and time coverage

Use these summaries before comparing positions. If a source has zero rows or missing lat/lon, edit `CONFIG["column_candidates"]` and rerun the notebook.

In [ ]:
def source_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame([{"source": name, "rows": 0}])

    return pd.DataFrame([{
        "source": name,
        "rows": len(df),
        "start": df["time_utc"].min(),
        "end": df["time_utc"].max(),
        "lat_non_null": df["lat"].notna().sum(),
        "lon_non_null": df["lon"].notna().sum(),
        "depth_non_null": df["depth"].notna().sum(),
        "altitude_non_null": df["altitude"].notna().sum(),
        "unique_vehicle_ids": df["vehicle_id"].nunique(),
    }])

summary = pd.concat([
    source_summary(gavia, "gavia"),
    source_summary(ecosub, "ecosub"),
    source_summary(sonardyne, "sonardyne"),
], ignore_index=True)

display(summary)

for name, df in [("gavia", gavia), ("ecosub", ecosub), ("sonardyne", sonardyne)]:
    if not df.empty:
        print(f"\\n{name} vehicle IDs / targets:")
        display(df["vehicle_id"].value_counts().head(20))

## 6. Coordinate handling

The comparison is done in a local metre-based coordinate frame.

- If lat/lon are available, the notebook uses `pyproj` to build a local azimuthal equidistant projection centred on the combined dataset.
- If `pyproj` is unavailable, it falls back to a good-enough local approximation for short ranges.
- If only easting/northing are available, those are used directly.

In [ ]:
def choose_origin(*dfs):
    coords = []
    for df in dfs:
        if df is not None and not df.empty and {"lat", "lon"}.issubset(df.columns):
            coords.append(df[["lat", "lon"]].dropna())

    if not coords:
        return None, None

    combined = pd.concat(coords, ignore_index=True)
    if combined.empty:
        return None, None

    return float(combined["lat"].median()), float(combined["lon"].median())


ORIGIN_LAT, ORIGIN_LON = choose_origin(gavia, ecosub, sonardyne)
print("Projection origin:", ORIGIN_LAT, ORIGIN_LON)


def add_local_xy(df: pd.DataFrame, origin_lat=ORIGIN_LAT, origin_lon=ORIGIN_LON) -> pd.DataFrame:
    df = df.copy()

    if df.empty:
        df["x_m"] = []
        df["y_m"] = []
        return df

    if df["lat"].notna().sum() == 0 or df["lon"].notna().sum() == 0:
        df["x_m"] = df["easting"]
        df["y_m"] = df["northing"]
        return df

    if origin_lat is None or origin_lon is None:
        df["x_m"] = np.nan
        df["y_m"] = np.nan
        return df

    valid = df["lat"].notna() & df["lon"].notna()
    df["x_m"] = np.nan
    df["y_m"] = np.nan

    if HAS_PYPROJ:
        crs_local = CRS.from_proj4(
            f"+proj=aeqd +lat_0={origin_lat} +lon_0={origin_lon} +datum=WGS84 +units=m +no_defs"
        )
        transformer = Transformer.from_crs("EPSG:4326", crs_local, always_xy=True)
        x, y = transformer.transform(df.loc[valid, "lon"].to_numpy(), df.loc[valid, "lat"].to_numpy())
        df.loc[valid, "x_m"] = x
        df.loc[valid, "y_m"] = y
    else:
        lat_rad = np.deg2rad(origin_lat)
        m_per_deg_lat = 111_320.0
        m_per_deg_lon = 111_320.0 * np.cos(lat_rad)
        df.loc[valid, "x_m"] = (df.loc[valid, "lon"] - origin_lon) * m_per_deg_lon
        df.loc[valid, "y_m"] = (df.loc[valid, "lat"] - origin_lat) * m_per_deg_lat

    return df


gavia_xy = add_local_xy(gavia)
ecosub_xy = add_local_xy(ecosub)
sonardyne_xy = add_local_xy(sonardyne)

display(gavia_xy.head())
display(ecosub_xy.head())
display(sonardyne_xy.head())

## 7. Basic track plots

These quick plots show whether the imported tracks occupy a plausible area before doing time-based matching.

In [ ]:
def plot_tracks_xy(*named_dfs):
    plt.figure(figsize=(9, 7))

    plotted = False
    for name, df in named_dfs:
        if df.empty or df["x_m"].notna().sum() == 0 or df["y_m"].notna().sum() == 0:
            continue
        plt.plot(df["x_m"], df["y_m"], marker=".", linestyle="-", markersize=2, linewidth=0.8, label=name)
        plotted = True

    if plotted:
        plt.axis("equal")
        plt.xlabel("East / local x (m)")
        plt.ylabel("North / local y (m)")
        plt.title("Vehicle and tracked positions")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("No plottable x/y positions found.")


plot_tracks_xy(("Gavia", gavia_xy), ("ecoSUB", ecosub_xy), ("Sonardyne/Ranger2", sonardyne_xy))

## 8. Time-align vehicle positions to Sonardyne/Ranger2 tracking

The default comparison uses nearest-neighbour time matching via `pandas.merge_asof`. The tolerance is set in `CONFIG["merge_tolerance"]`.

If Ranger2 exports contain multiple targets in one file, filter `sonardyne` by `vehicle_id` before comparing, or run comparisons separately by beacon/target.

In [ ]:
def prepare_for_asof(df: pd.DataFrame, prefix: str | None = None) -> pd.DataFrame:
    cols = ["time_utc", "lat", "lon", "depth", "altitude", "x_m", "y_m", "vehicle_id", "source_file"]
    cols = [c for c in cols if c in df.columns]
    out = df[cols].dropna(subset=["time_utc"]).sort_values("time_utc").copy()
    if prefix:
        rename = {c: f"{prefix}{c}" for c in cols if c != "time_utc"}
        out = out.rename(columns=rename)
    return out


def compare_vehicle_to_track(
    vehicle_df: pd.DataFrame,
    track_df: pd.DataFrame,
    vehicle_name: str,
    tolerance=CONFIG["merge_tolerance"],
) -> pd.DataFrame:
    if vehicle_df.empty:
        warnings.warn(f"{vehicle_name}: no vehicle rows to compare.")
        return pd.DataFrame()

    if track_df.empty:
        warnings.warn("No Sonardyne/Ranger2 rows to compare against.")
        return pd.DataFrame()

    veh = prepare_for_asof(vehicle_df).rename(columns={
        "source_file": "vehicle_source_file",
    })

    trk = prepare_for_asof(track_df, prefix="track_")

    matched = pd.merge_asof(
        veh,
        trk,
        on="time_utc",
        direction="nearest",
        tolerance=pd.Timedelta(tolerance),
    )

    matched["source_vehicle"] = vehicle_name

    trk_time = track_df[["time_utc"]].sort_values("time_utc").rename(columns={"time_utc": "track_time_utc"})
    matched = pd.merge_asof(
        matched.sort_values("time_utc"),
        trk_time,
        left_on="time_utc",
        right_on="track_time_utc",
        direction="nearest",
        tolerance=pd.Timedelta(tolerance),
    )

    matched["dt_seconds"] = (matched["time_utc"] - matched["track_time_utc"]).dt.total_seconds()

    matched["east_error_m"] = matched["x_m"] - matched["track_x_m"]
    matched["north_error_m"] = matched["y_m"] - matched["track_y_m"]
    matched["horizontal_error_m"] = np.hypot(matched["east_error_m"], matched["north_error_m"])

    if "depth" in matched.columns and "track_depth" in matched.columns:
        matched["depth_error_m"] = matched["depth"] - matched["track_depth"]

    return matched


comparisons = []
if not gavia_xy.empty:
    comparisons.append(compare_vehicle_to_track(gavia_xy, sonardyne_xy, "gavia"))
if not ecosub_xy.empty:
    comparisons.append(compare_vehicle_to_track(ecosub_xy, sonardyne_xy, "ecosub"))

comparison = pd.concat([c for c in comparisons if not c.empty], ignore_index=True) if comparisons else pd.DataFrame()

print(f"Matched comparison rows: {len(comparison):,}")
display(comparison.head())

## 9. Error statistics and plots

The key QC metric is `horizontal_error_m`, but always check `dt_seconds` too. Large position errors can be caused by poor time synchronisation rather than poor tracking/navigation.

In [ ]:
def error_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    matched = df.dropna(subset=["horizontal_error_m"])
    if matched.empty:
        return pd.DataFrame()

    groups = []
    for source, g in matched.groupby("source_vehicle"):
        groups.append({
            "source_vehicle": source,
            "matched_rows": len(g),
            "median_horizontal_error_m": g["horizontal_error_m"].median(),
            "mean_horizontal_error_m": g["horizontal_error_m"].mean(),
            "p95_horizontal_error_m": g["horizontal_error_m"].quantile(0.95),
            "max_horizontal_error_m": g["horizontal_error_m"].max(),
            "median_abs_dt_seconds": g["dt_seconds"].abs().median(),
            "p95_abs_dt_seconds": g["dt_seconds"].abs().quantile(0.95),
        })

    return pd.DataFrame(groups).sort_values("source_vehicle")


stats = error_summary(comparison)
display(stats)

In [ ]:
def plot_error_timeseries(df: pd.DataFrame):
    if df.empty or "horizontal_error_m" not in df.columns:
        print("No comparison data to plot.")
        return

    for source, g in df.dropna(subset=["horizontal_error_m"]).groupby("source_vehicle"):
        plt.figure(figsize=(11, 4))
        plt.plot(g["time_utc"], g["horizontal_error_m"], marker=".", linestyle="-", markersize=2, linewidth=0.8)
        plt.xlabel("Time UTC")
        plt.ylabel("Horizontal error (m)")
        plt.title(f"{source}: vehicle position vs Sonardyne/Ranger2")
        plt.grid(True)
        plt.show()


def plot_error_histogram(df: pd.DataFrame):
    if df.empty or "horizontal_error_m" not in df.columns:
        print("No comparison data to plot.")
        return

    for source, g in df.dropna(subset=["horizontal_error_m"]).groupby("source_vehicle"):
        plt.figure(figsize=(8, 4))
        plt.hist(g["horizontal_error_m"], bins=50)
        plt.xlabel("Horizontal error (m)")
        plt.ylabel("Count")
        plt.title(f"{source}: horizontal error distribution")
        plt.grid(True)
        plt.show()


plot_error_timeseries(comparison)
plot_error_histogram(comparison)

In [ ]:
def plot_error_vectors(df: pd.DataFrame, every=10):
    if df.empty:
        print("No comparison data to plot.")
        return

    for source, g in df.dropna(subset=["x_m", "y_m", "track_x_m", "track_y_m"]).groupby("source_vehicle"):
        g = g.iloc[::every].copy()
        if g.empty:
            continue

        plt.figure(figsize=(9, 7))
        plt.plot(g["track_x_m"], g["track_y_m"], ".", markersize=2, label="Ranger2/Sonardyne")
        plt.plot(g["x_m"], g["y_m"], ".", markersize=2, label=source)
        plt.quiver(
            g["track_x_m"], g["track_y_m"],
            g["x_m"] - g["track_x_m"], g["y_m"] - g["track_y_m"],
            angles="xy", scale_units="xy", scale=1, width=0.002
        )
        plt.axis("equal")
        plt.xlabel("East / local x (m)")
        plt.ylabel("North / local y (m)")
        plt.title(f"{source}: error vectors, every {every}th matched fix")
        plt.legend()
        plt.grid(True)
        plt.show()


plot_error_vectors(comparison, every=20)

## 10. Export cleaned products

These outputs are useful for archiving and further analysis:

- `gavia_positions_standardised.csv`
- `ecosub_positions_standardised.csv`
- `sonardyne_positions_standardised.csv`
- `vehicle_vs_sonardyne_comparison.csv`
- `vehicle_vs_sonardyne_error_summary.csv`

In [ ]:
OUT = CONFIG["paths"]["output_dir"]
OUT.mkdir(parents=True, exist_ok=True)

gavia_xy.to_csv(OUT / "gavia_positions_standardised.csv", index=False)
ecosub_xy.to_csv(OUT / "ecosub_positions_standardised.csv", index=False)
sonardyne_xy.to_csv(OUT / "sonardyne_positions_standardised.csv", index=False)

if not comparison.empty:
    comparison.to_csv(OUT / "vehicle_vs_sonardyne_comparison.csv", index=False)

if not stats.empty:
    stats.to_csv(OUT / "vehicle_vs_sonardyne_error_summary.csv", index=False)

print(f"Saved outputs to: {OUT}")

## 11. Troubleshooting and tuning

### If Gavia loads zero useful rows

Open one of the intermediate flattened CSV files in `output/gavia_flat_csv/` and inspect the column names:

```python
raw = read_csv_flexible(gavia_flat_csvs[0])
raw.head()
raw.columns.tolist()
```

Then add the correct names to `CONFIG["column_candidates"]`.

### If Sonardyne/Ranger2 has multiple tracked targets

Check target names:

```python
sonardyne["vehicle_id"].value_counts()
```

Then either set a filter in `CONFIG["vehicle_name_filters"]["sonardyne"]` or split the tracking table by `vehicle_id`.

### If errors are consistently large

Check:

1. timestamp timezone and clock drift,
2. Ranger2 export coordinate system,
3. whether vehicle GPS fixes are surface-only while acoustic fixes continue underwater,
4. whether Gavia/ecoSUB positions are dead-reckoned, GPS, or USBL-aided,
5. whether lat/lon are decimal degrees or degree/minute strings.